In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls -lhL /content/drive/MyDrive/bridge_project

total 913M
-r-------- 1 root root 4.6M Nov  1 13:23 all_polygons.csv
-r-------- 1 root root  94M Nov  3 06:58 baseline_unet_r34_FINAL.pth
-r-------- 1 root root  103 Nov  3 06:56 best_thresholds.json
-r-------- 1 root root  94M Nov  2 10:36 best_unet_r34.pth
dr-x------ 2 root root 4.0K Nov  1 13:23 figures
-r-------- 1 root root 3.0K Nov  2 10:36 history.csv
-r-------- 1 root root 722M Nov  1 13:46 processed.zip


In [3]:
!unzip -q /content/drive/MyDrive/bridge_project/processed.zip -d /content/
!ls /content/processed
!pip install -q segmentation-models-pytorch albumentations opencv-python-headless scikit-image

test  train  val


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, random, re, time
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

PROC = Path("/content/processed")
BASE = Path("/content/drive/MyDrive/bridge_project")          # shared, read-only
EXP_ROOT = Path("/content/drive/MyDrive/bridge_experiments")  # YOUR own results folder
TARGET = ["Crack", "Spalling", "Rust", "Efflorescence", "Cavity", "ExposedRebars"]
NC = len(TARGET); SIZE = 512
DEV = "cuda" if torch.cuda.is_available() else "cpu"
COLORS = [(255,0,0),(0,255,0),(0,0,255),(255,255,0),(255,0,255),(0,255,255)]

def decode(m): return np.stack([(m >> c) & 1 for c in range(NC)], -1).astype(np.float32)
def seed_everything(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything()

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
tf_train = A.Compose([A.HorizontalFlip(p=0.5), A.Affine(scale=(0.85, 1.15), rotate=(-10, 10), p=0.5),
                      A.RandomBrightnessContrast(p=0.5), A.HueSaturationValue(p=0.3),
                      A.Normalize(MEAN, STD), ToTensorV2(transpose_mask=True)])
tf_val = A.Compose([A.Normalize(MEAN, STD), ToTensorV2(transpose_mask=True)])

class DaclDataset(Dataset):
    def __init__(self, split, tf):
        self.split, self.tf = split, tf
        self.ids = sorted(p.stem for p in (PROC/split/"img").glob("*.jpg"))
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        s = self.ids[i]
        img = cv2.cvtColor(cv2.imread(str(PROC/self.split/"img"/f"{s}.jpg")), cv2.COLOR_BGR2RGB)
        m = cv2.imread(str(PROC/self.split/"mask"/f"{s}.png"), cv2.IMREAD_GRAYSCALE)
        out = self.tf(image=img, mask=decode(m))
        return out["image"], out["mask"]

train_dl = DataLoader(DaclDataset("train", tf_train), batch_size=8, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
val_dl   = DataLoader(DaclDataset("val",   tf_val),   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(DaclDataset("test",  tf_val),   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

bce = nn.BCEWithLogitsLoss()
dice = smp.losses.DiceLoss(mode="multilabel", from_logits=True)
def loss_fn(logits, y): return bce(logits, y) + dice(logits, y)

@torch.no_grad()
def evaluate(model, loader, thr=0.5):
    model.eval()
    thr = torch.as_tensor(thr, device=DEV, dtype=torch.float32).view(1, -1, 1, 1) if not np.isscalar(thr) else thr
    tp = torch.zeros(NC, device=DEV); fp = torch.zeros(NC, device=DEV); fn = torch.zeros(NC, device=DEV)
    for x, y in loader:
        x, y = x.to(DEV), y.to(DEV)
        with torch.autocast("cuda"):
            p = torch.sigmoid(model(x))
        pr = (p > thr).float()
        tp += (pr * y).sum((0, 2, 3)); fp += (pr * (1 - y)).sum((0, 2, 3)); fn += ((1 - pr) * y).sum((0, 2, 3))
    iou = tp / (tp + fp + fn + 1e-7); dsc = 2 * tp / (2 * tp + fp + fn + 1e-7)
    prec = tp / (tp + fp + 1e-7); rec = tp / (tp + fn + 1e-7)
    return {k: v.cpu().numpy() for k, v in dict(iou=iou, dice=dsc, precision=prec, recall=rec).items()}

th = json.load(open(BASE/"best_thresholds.json"))
best_thr_baseline = np.array([th[t] for t in TARGET])
print({s: len(list((PROC/s/"img").glob("*.jpg"))) for s in ["train", "val", "test"]}, "| GPU:", torch.cuda.is_available())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'train': 5245, 'val': 582, 'test': 975} | GPU: True


In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("GPU:", torch.cuda.is_available())

Device: cuda
GPU: True


In [4]:
base_model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=NC, activation=None).to(DEV)
base_model.load_state_dict(torch.load(BASE/"best_unet_r34.pth", map_location=DEV)); base_model.eval()
r = evaluate(base_model, val_dl, thr=best_thr_baseline)
print("baseline val mIoU:", round(float(r["iou"].mean()), 3))
del base_model; torch.cuda.empty_cache()

baseline val mIoU: 0.326


In [6]:
EXP = "fpn_r34"
model = smp.FPN(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=NC, activation=None).to(DEV)

OUT = EXP_ROOT/EXP
OUT.mkdir(parents=True, exist_ok=True)
CKPT = OUT/"best.pth"
print(EXP, "-> results go to:", OUT, "| parameters (millions):", round(sum(p.numel() for p in model.parameters())/1e6, 1))

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

fpn_r34 -> results go to: /content/drive/MyDrive/bridge_experiments/fpn_r34 | parameters (millions): 23.2


In [8]:
EPOCHS = 30
LAST = OUT / "last.pth"

opt = torch.optim.AdamW(
    model.parameters(), lr=3e-4, weight_decay=1e-4
)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda")

hist, best, start = [], 0.0, 0

if LAST.exists():
    ck = torch.load(LAST, map_location=DEV, weights_only=False)
    model.load_state_dict(ck["model"])
    opt.load_state_dict(ck["opt"])
    sched.load_state_dict(ck["sched"])
    scaler.load_state_dict(ck["scaler"])
    hist, best, start = ck["hist"], ck["best"], ck["epoch"] + 1
    print(f"RESUMING: epochs 0 to {start-1} are already done.")
else:
    print("Starting from epoch 0.")

for ep in range(start, EPOCHS):
    t0 = time.time()
    model.train()
    tot = 0.0

    for x, y in train_dl:
        x, y = x.to(DEV), y.to(DEV)
        opt.zero_grad()

        with torch.autocast("cuda"):
            loss = loss_fn(model(x), y)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        tot += loss.item()

    sched.step()
    m = evaluate(model, val_dl)
    miou = float(m["iou"].mean())

    hist.append(dict(
        epoch=ep,
        train_loss=tot / len(train_dl),
        val_miou=miou,
        minutes=(time.time()-t0)/60,
        **{f"iou_{t}": float(v) for t, v in zip(TARGET, m["iou"])}
    ))

    print(
        f"ep{ep:02d} loss {tot/len(train_dl):.3f} "
        f"val mIoU {miou:.3f}",
        m["iou"].round(3),
        f"{(time.time()-t0)/60:.1f} min",
        flush=True
    )

    if miou > best:
        best = miou
        torch.save(model.state_dict(), CKPT)

    pd.DataFrame(hist).to_csv(OUT / "history.csv", index=False)

    torch.save(dict(
        model=model.state_dict(),
        opt=opt.state_dict(),
        sched=sched.state_dict(),
        scaler=scaler.state_dict(),
        hist=hist,
        best=best,
        epoch=ep
    ), LAST)

print("TRAINING COMPLETE. Best validation mIoU:", round(best, 3))

Starting from epoch 0.
ep00 loss 0.831 val mIoU 0.197 [0.14  0.234 0.267 0.207 0.174 0.16 ] 2.8 min
ep01 loss 0.749 val mIoU 0.229 [0.198 0.258 0.354 0.229 0.136 0.197] 3.1 min
ep02 loss 0.729 val mIoU 0.246 [0.215 0.292 0.364 0.234 0.171 0.203] 3.3 min
ep03 loss 0.706 val mIoU 0.257 [0.215 0.305 0.36  0.299 0.137 0.227] 3.2 min
ep04 loss 0.700 val mIoU 0.277 [0.247 0.332 0.376 0.32  0.138 0.247] 3.1 min
ep05 loss 0.678 val mIoU 0.285 [0.239 0.301 0.396 0.328 0.198 0.248] 3.2 min
ep06 loss 0.666 val mIoU 0.269 [0.252 0.328 0.366 0.258 0.146 0.264] 3.1 min
ep07 loss 0.650 val mIoU 0.276 [0.223 0.307 0.424 0.293 0.176 0.234] 3.1 min
ep08 loss 0.649 val mIoU 0.281 [0.246 0.337 0.392 0.326 0.132 0.252] 3.1 min
ep09 loss 0.636 val mIoU 0.285 [0.244 0.314 0.39  0.337 0.156 0.271] 3.1 min
ep10 loss 0.627 val mIoU 0.291 [0.259 0.362 0.41  0.311 0.172 0.234] 3.2 min
ep11 loss 0.611 val mIoU 0.291 [0.248 0.352 0.371 0.302 0.191 0.282] 3.1 min
ep12 loss 0.604 val mIoU 0.274 [0.247 0.348 0.318 0.3

In [9]:
model.load_state_dict(torch.load(CKPT, map_location=DEV))
model.eval()

ths = np.arange(0.1, 0.91, 0.1)
res = np.zeros((len(ths), NC))

for i, t in enumerate(ths):
    res[i] = evaluate(model, val_dl, thr=float(t))["iou"]

thr_exp = ths[res.argmax(0)]

test_tuned = evaluate(model, test_dl, thr=thr_exp)
test_050 = evaluate(model, test_dl, thr=0.5)

h = pd.read_csv(OUT / "history.csv")

summary = dict(
    experiment=EXP,
    best_epoch=int(h.val_miou.idxmax()),
    val_miou_thr050=float(h.val_miou.max()),
    val_miou_tuned=float(res.max(0).mean()),
    test_miou_tuned=float(test_tuned["iou"].mean()),
    test_miou_thr050=float(test_050["iou"].mean()),
    test_iou_per_class=dict(
        zip(TARGET, [round(float(v), 3) for v in test_tuned["iou"]])
    ),
    thresholds=dict(
        zip(TARGET, [round(float(v), 2) for v in thr_exp])
    ),
    minutes_per_epoch=round(float(h.minutes.mean()), 2),
    params_millions=round(
        sum(p.numel() for p in model.parameters()) / 1e6, 1
    ),
)

json.dump(summary, open(OUT / "results.json", "w"), indent=2)

print(json.dumps(summary, indent=2))

{
  "experiment": "fpn_r34",
  "best_epoch": 19,
  "val_miou_thr050": 0.3299059867858886,
  "val_miou_tuned": 0.3386680136124293,
  "test_miou_tuned": 0.30561161041259766,
  "test_miou_thr050": 0.3112276494503021,
  "test_iou_per_class": {
    "Crack": 0.249,
    "Spalling": 0.374,
    "Rust": 0.37,
    "Efflorescence": 0.312,
    "Cavity": 0.147,
    "ExposedRebars": 0.381
  },
  "thresholds": {
    "Crack": 0.7,
    "Spalling": 0.4,
    "Rust": 0.2,
    "Efflorescence": 0.8,
    "Cavity": 0.1,
    "ExposedRebars": 0.1
  },
  "minutes_per_epoch": 3.16,
  "params_millions": 23.2
}
